# TASK 3: RAG (Retrieval-Augmented Generation) with Unsloth
# Platform: Google Colab (T4 GPU)

In [1]:
# ──────────────────────────────────────────
# CELL 1: Install Libraries
# ──────────────────────────────────────────
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install faiss-cpu sentence-transformers langchain langchain-community

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-l_ml_zql/unsloth_a682eb941c6144e9a6fa10b7ec46b0cf
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-l_ml_zql/unsloth_a682eb941c6144e9a6fa10b7ec46b0cf
  Resolved https://github.com/unslothai/unsloth.git to commit 7ef8cde3c22a3bc2240353353caa21051a42ea90
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 22.8 MB/s eta 0:00:00

In [2]:
# ──────────────────────────────────────────
# CELL 2: Import Libraries
# ──────────────────────────────────────────

import torch
import numpy as np
from unsloth import FastLanguageModel
from sentence_transformers import SentenceTransformer
import faiss                           # Facebook's fast similarity search library

print("All libraries imported!")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
All libraries imported!
GPU: Tesla T4


In [3]:
# ──────────────────────────────────────────
# CELL 3: Create Medical Knowledge Base
# ──────────────────────────────────────────

"""
This is our "document library" - the knowledge the RAG will search through.
In real use, you'd load PDFs, text files, etc.
"""

# Medical knowledge documents (knowledge base)
medical_documents = [
    """
    Diabetes Mellitus Type 2: This is a chronic metabolic disorder where the body
    does not properly use insulin. Symptoms include increased thirst, frequent
    urination, fatigue, blurred vision, and slow-healing wounds.
    Treatment includes lifestyle changes (diet, exercise), oral medications like
    metformin, and sometimes insulin injections. Regular blood glucose monitoring
    is essential. HbA1c target is below 7% for most patients.
    """,

    """
    Hypertension (High Blood Pressure): Blood pressure consistently above 130/80 mmHg.
    It is called the 'silent killer' because it has no symptoms.
    Risk factors include obesity, salt intake, stress, family history, and age.
    Treatment: ACE inhibitors, beta-blockers, calcium channel blockers, diuretics.
    Lifestyle changes: reduce salt, exercise regularly, quit smoking, limit alcohol.
    Target blood pressure: below 130/80 mmHg for most adults.
    """,

    """
    Myocardial Infarction (Heart Attack): Occurs when blood supply to part of heart is blocked.
    Symptoms: chest pain/pressure, pain radiating to arm/jaw, shortness of breath,
    sweating, nausea, dizziness.
    Emergency treatment: call ambulance immediately. Give aspirin 325mg if not allergic.
    Hospital treatment: thrombolysis or PCI (stent placement).
    Risk factors: smoking, diabetes, hypertension, high cholesterol, family history.
    """,

    """
    Asthma: A chronic respiratory disease causing airway inflammation and narrowing.
    Symptoms: wheezing, shortness of breath, chest tightness, coughing (especially at night).
    Triggers: dust mites, pollen, pet dander, cold air, exercise, smoke.
    Treatment:
    - Reliever inhalers: Short-acting beta-agonists (salbutamol) for immediate relief
    - Preventer inhalers: Corticosteroids (beclomethasone) for daily use
    - Severe cases: oral steroids, biologics (omalizumab)
    """,

    """
    Pneumonia: Infection of the lungs, can be bacterial, viral, or fungal.
    Common causes: Streptococcus pneumoniae (most common bacterial cause).
    Symptoms: fever, productive cough, chest pain, difficulty breathing, fatigue.
    Diagnosis: chest X-ray, sputum culture, blood tests.
    Treatment: antibiotics (amoxicillin for mild cases), hospital admission for severe cases.
    Prevention: pneumococcal vaccine, influenza vaccine.
    """,

    """
    Chronic Kidney Disease (CKD): Progressive loss of kidney function.
    Stages: 1-5 based on GFR (Glomerular Filtration Rate).
    Causes: diabetes (most common), hypertension, glomerulonephritis.
    Symptoms in early stages: none. Late stages: fatigue, swelling, nausea, uremia.
    Treatment: treat underlying cause, ACE inhibitors, manage blood pressure and blood sugar.
    Stage 5 (kidney failure): dialysis or kidney transplant required.
    """,

    """
    Influenza (Flu): Contagious respiratory illness caused by influenza viruses.
    Symptoms: sudden onset of fever, body aches, headache, fatigue, dry cough, sore throat.
    Difference from common cold: flu comes on suddenly, more severe, causes fever and body aches.
    Treatment: rest, fluids, antiviral (oseltamivir/Tamiflu) within 48 hours.
    Prevention: annual flu vaccine recommended for everyone above 6 months.
    Complications: pneumonia, especially in elderly, young children, immunocompromised.
    """,

    """
    Stroke: Medical emergency - blood supply to brain is interrupted.
    Types: Ischemic (clot, 85% of strokes) and Hemorrhagic (bleeding, 15%).
    FAST test: Face drooping, Arm weakness, Speech difficulty, Time to call emergency.
    Treatment: 'Time is brain' - thrombolysis within 4.5 hours for ischemic stroke.
    Risk factors: hypertension, atrial fibrillation, diabetes, high cholesterol, smoking.
    Prevention: control blood pressure, anticoagulants for AFib, lifestyle changes.
    """,
]

print(f" Knowledge base created with {len(medical_documents)} documents")


 Knowledge base created with 8 documents


In [4]:
# ──────────────────────────────────────────
# CELL 4: Create Embeddings (Turn text into numbers)
# ──────────────────────────────────────────

"""
Embeddings = Converting text to numbers (vectors) that capture meaning.
Similar topics will have similar numbers, so we can find relevant documents.
"""

print("Loading embedding model...")

# Load a sentence transformer for creating embeddings
# This is a small, fast model for encoding text
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Creating embeddings for all documents...")

# Convert each document to a vector of numbers
document_embeddings = embedding_model.encode(
    medical_documents,
    show_progress_bar=True,
    normalize_embeddings=True   # Normalize for cosine similarity
)

print(f"Embeddings created! Shape: {document_embeddings.shape}")
print(f"Each document is represented as {document_embeddings.shape[1]} numbers")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Creating embeddings for all documents...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings created! Shape: (8, 384)
Each document is represented as 384 numbers


In [5]:
# ──────────────────────────────────────────
# CELL 5: Build FAISS Search Index
# ──────────────────────────────────────────

"""
FAISS = Fast Approximate Index for Similarity Search
It lets us quickly find the most similar documents to a query.
"""

# Get the size of our embedding vectors
embedding_size = document_embeddings.shape[1]

# Create a FAISS index (uses cosine similarity)
index = faiss.IndexFlatIP(embedding_size)  # IP = Inner Product (for normalized = cosine)

# Add all document embeddings to the index
index.add(document_embeddings.astype(np.float32))

print(f"FAISS index built with {index.ntotal} documents")

FAISS index built with 8 documents


In [6]:
# ──────────────────────────────────────────
# CELL 6: Create Retrieval Function
# ──────────────────────────────────────────

def retrieve_relevant_docs(query, top_k=2):
    """
    Finds the most relevant documents for a given query.

    Parameters:
        query: the user's question
        top_k: how many documents to return

    Returns:
        List of the most relevant document texts
    """
    # Convert query to embedding
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype(np.float32)

    # Search for top_k most similar documents
    distances, indices = index.search(query_embedding, top_k)

    # Return the actual document texts
    retrieved = []
    for i, idx in enumerate(indices[0]):
        score = distances[0][i]
        retrieved.append({
            "document": medical_documents[idx].strip(),
            "similarity_score": float(score),
            "rank": i + 1
        })

    return retrieved

# Test retrieval
print("Testing retrieval...")
test_results = retrieve_relevant_docs("What causes high blood pressure?")
print(f"Found {len(test_results)} relevant documents")
print(f"Top result similarity: {test_results[0]['similarity_score']:.3f}")
print(f"Top result preview: {test_results[0]['document'][:150]}...")


Testing retrieval...
Found 2 relevant documents
Top result similarity: 0.519
Top result preview: Hypertension (High Blood Pressure): Blood pressure consistently above 130/80 mmHg.
    It is called the 'silent killer' because it has no symptoms.
  ...


In [8]:
import torch
import numpy as np
from unsloth import FastLanguageModel
from sentence_transformers import SentenceTransformer
import faiss                           # Facebook's fast similarity search library



# ──────────────────────────────────────────
# CELL 7: Load Quantized LLM
# ──────────────────────────────────────────

print("Loading language model (this may take 2-3 minutes)...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit", # Changed model name
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,    # 4-bit quantization = less memory
)

# Set to inference mode for faster generation
FastLanguageModel.for_inference(model)

print(" LLM loaded and ready!")

Loading language model (this may take 2-3 minutes)...
==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


 LLM loaded and ready!


In [9]:
# ──────────────────────────────────────────
# CELL 8: Create RAG Answer Function
# ──────────────────────────────────────────

def rag_answer(question, top_k=2):
    """
    Full RAG pipeline:
    1. Retrieve relevant documents
    2. Create a prompt with context + question
    3. Generate answer from LLM

    Parameters:
        question: the user's medical question
        top_k: number of documents to retrieve

    Returns:
        Generated answer string
    """

    print(f"\nQuestion: {question}")
    print("-" * 50)

    # STEP 1: Retrieve relevant documents
    relevant_docs = retrieve_relevant_docs(question, top_k=top_k)

    # STEP 2: Build context string from retrieved docs
    context = "\n\n".join([
        f"Document {doc['rank']} (relevance: {doc['similarity_score']:.2f}):\n{doc['document']}"
        for doc in relevant_docs
    ])

    print(f"Retrieved {len(relevant_docs)} relevant documents")

    # STEP 3: Create RAG prompt (context + question)
    rag_prompt = f"""You are a helpful medical assistant. Use the provided medical context to answer the question accurately.

MEDICAL CONTEXT:
{context}

QUESTION: {question}

ANSWER: Based on the medical information provided, """

    # STEP 4: Tokenize the prompt
    inputs = tokenizer(
        [rag_prompt],
        return_tensors="pt"
    ).to("cuda")

    # STEP 5: Generate answer
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            temperature=0.7,
            do_sample=True,
            use_cache=True,
        )

    # STEP 6: Decode and clean the response
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the answer part (after "ANSWER:")
    if "ANSWER:" in full_response:
        answer = full_response.split("ANSWER:")[-1].strip()
    else:
        answer = full_response[len(rag_prompt):].strip()

    return answer, relevant_docs

In [10]:
# ──────────────────────────────────────────
# CELL 9: Test RAG System
# ──────────────────────────────────────────

# Test questions
test_questions = [
    "What are the symptoms of a heart attack?",
    "How is diabetes treated?",
    "What is the FAST test for stroke?",
    "What medications are used for high blood pressure?",
]

for question in test_questions:
    print("\n" + "="*60)
    answer, docs = rag_answer(question)
    print(f"\n ANSWER:\n{answer[:400]}")
    print(f"\n Retrieved from {len(docs)} document(s)")
    print(f"   Top doc similarity: {docs[0]['similarity_score']:.3f}")



Both `max_new_tokens` (=250) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




Question: What are the symptoms of a heart attack?
--------------------------------------------------
Retrieved 2 relevant documents


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i


 ANSWER:
Based on the medical information provided,  the symptoms of a heart attack (myocardial infarction) include:

* Chest pain or pressure
* Pain radiating to the arm or jaw
* Shortness of breath
* Sweating
* Nausea
* Dizziness

These symptoms are considered emergency situations and require immediate medical attention. If not allergic, aspirin 325mg should be given and an ambulance should be called. Fu

 Retrieved from 2 document(s)
   Top doc similarity: 0.662


Question: How is diabetes treated?
--------------------------------------------------
Retrieved 2 relevant documents


Both `max_new_tokens` (=250) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 ANSWER:
Based on the medical information provided,  diabetes (Type 2) is treated through lifestyle changes (diet, exercise), oral medications like metformin, and sometimes insulin injections. Regular blood glucose monitoring is essential, with a target HbA1c level of below 7% for most patients. This is in accordance with Document 1. The treatment information is not related to the condition of Chronic Kidn

 Retrieved from 2 document(s)
   Top doc similarity: 0.591


Question: What is the FAST test for stroke?
--------------------------------------------------
Retrieved 2 relevant documents


Both `max_new_tokens` (=250) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 ANSWER:
Based on the medical information provided,  the FAST test is a simple test used to identify potential signs of a stroke. The acronym stands for:

F - Face: Does one side of the face droop or have a different appearance compared to the other side?
A - Arm: Is one arm weak or numb, making it difficult to lift or move?
S - Speech: Is speech difficulty or slurred speech observed?
T - Time: Time is bra

 Retrieved from 2 document(s)
   Top doc similarity: 0.599


Question: What medications are used for high blood pressure?
--------------------------------------------------
Retrieved 2 relevant documents

 ANSWER:
Based on the medical information provided,  ACE inhibitors, beta-blockers, calcium channel blockers, and diuretics are medications used for high blood pressure (hypertension). These medications are mentioned in Document 1 as treatment options for hypertension. 

Please provide any additional information or clarify any doubts. Thank you for your time!

 Retrieved from 2 do

In [11]:
# ──────────────────────────────────────────
# CELL 10: Compare RAG vs No-RAG
# ──────────────────────────────────────────

"""
Let's show the difference between answering WITH and WITHOUT context.
This proves RAG helps the model give better, grounded answers.
"""

question = "What medications are used to treat hypertension?"

# ── Without RAG (no context) ──
prompt_no_rag = f"Question: {question}\nAnswer:"
inputs = tokenizer([prompt_no_rag], return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)
no_rag_answer = tokenizer.decode(outputs[0], skip_special_tokens=True).split("Answer:")[-1].strip()

# ── With RAG ──
rag_result, _ = rag_answer(question)

print("\n" + "="*60)
print("COMPARISON: RAG vs No-RAG")
print("="*60)
print(f"\nQuestion: {question}")
print(f"\n WITHOUT RAG:\n{no_rag_answer[:300]}")
print(f"\n WITH RAG:\n{rag_result[:300]}")


Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=250) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question: What medications are used to treat hypertension?
--------------------------------------------------
Retrieved 2 relevant documents


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



COMPARISON: RAG vs No-RAG

Question: What medications are used to treat hypertension?

 WITHOUT RAG:
There are many medications that are used to treat hypertension. Some of the most common include:

1. Diuretics: These medications help remove excess fluid and sodium from the body by increasing urine production. Examples include hydrochlorothiazide (HCTZ) and furosemide.
2. Beta blockers: These medi

 WITH RAG:
Based on the medical information provided,  ACE inhibitors, beta-blockers, calcium channel blockers, and diuretics are used to treat hypertension. These medications are mentioned in Document 1, which is relevant to the topic of hypertension. (Relevance score: 0.38) There is no mention of these medic
